In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict

In [4]:
load_dotenv()

True

In [5]:
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    task = "text-generation"
)

model = ChatHuggingFace(llm=llm)

c:\Users\sodiu\OneDrive\Documents\Coding\LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [7]:
def llm_qa(state: LLMState) -> LLMState:
    #extratct the question from state
    question = state['question']

    #form a prompt
    prompt = f'Answer the following question {question}'

    #ask that question to the LLM
    answer = model.invoke(prompt).content

    #update the answer in the state
    state['answer'] = answer

    return state

In [11]:
#create Curr graph
graph = StateGraph(LLMState)

#add Nodes
graph.add_node('llm_qa', llm_qa)

#add Edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

#compile graph
workflow = graph.compile()

In [12]:
#excecute
initial_state = {'question': 'How far is the moon from earth'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'How far is the moon from earth', 'answer': "The average distance from the Earth to the Moon is about 384,400 kilometers (238,900 miles). This distance can vary slightly due to the elliptical shape of the Moon's orbit around our planet. At its closest point (called perigee), the Moon is about 363,300 kilometers (225,000 miles) away, and at its farthest point (called apogee), it is about 405,500 kilometers (252,000 miles) away.\n\nHere's a rough breakdown of the average distance from the Earth to the Moon:\n\n* Average distance: 384,400 kilometers (238,900 miles)\n* Perigee (closest point): 363,300 kilometers (225,000 miles)\n* Apogee (farthest point): 405,500 kilometers (252,000 miles)\n\nKeep in mind that these distances are constantly changing due to the Moon's elliptical orbit, which is influenced by the gravitational pull of both the Earth and the Sun."}
